# 03 — Submission
Memuat model dari `models/pipeline_model_lgbm.pkl`, memprediksi `test.csv`, dan menulis
`submissions/submission.csv` dengan format dan urutan baris yang sama persis dengan `sample_submission.csv`.

Di Kaggle, file juga disalin ke `/kaggle/working/submission.csv`. Jika notebook ini dijalankan di sesi
terpisah dari `02_modeling.ipynb`, tambahkan output notebook 02 (yang berisi file `.pkl`) sebagai *Input*;
fungsi `find_model` akan mencarinya di `/kaggle/input`.

In [ ]:
# === Setup: temukan folder repo (lokal / Kaggle) ===
import sys, subprocess
from pathlib import Path

# Jika dijalankan di Kaggle dan repo belum ada, isi URL GitHub repo kamu (Internet harus ON):
REPO_URL = ""   # contoh: "https://github.com/username/structural-damage-classification.git"

def find_repo_root():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/repo")]
    kin = Path("/kaggle/input")
    if kin.exists():
        cands += [p.parent.parent for p in kin.glob("**/scripts/__init__.py")]
    for c in cands:
        if (c / "scripts" / "__init__.py").exists():
            return c.resolve()
    return None

ROOT = find_repo_root()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    ROOT = Path("/kaggle/working/repo")
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan. Isi REPO_URL atau upload repo sebagai Kaggle Dataset."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import pandas as pd
from scripts.preprocess import load_data, TARGET, ID_COL
from scripts.predict import load_model, predict_grades, format_submission, save_submission

model = load_model()
_, test, sample = load_data()
grades = predict_grades(model, test)
sub = format_submission(sample, test[ID_COL], grades)
path = save_submission(sub)
print("tersimpan:", path, sub.shape)
sub.head()

In [ ]:
# Validasi format
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) and (sub[ID_COL].to_numpy() == sample[ID_COL].to_numpy()).all()
assert sub[TARGET].notna().all()
print("format OK")
print(sub[TARGET].value_counts(normalize=True).sort_index().round(3))